In [25]:
import ast
import base64
import contextlib
import io
from typing import Dict, Tuple, Optional

import matplotlib.pyplot as plt


def execute_with_capture(code_str: str, global_env: Optional[Dict[str, object]] = None, execution_count: int = 1) -> Tuple[Dict[str, object], Dict[str, object], int]:
    """Execute code the way Jupyter does, but capture stdout/stderr and matplotlib output."""
    global_env = dict(global_env or {})
    global_env.setdefault('__builtins__', __builtins__)
    global_env.setdefault('plt', plt)

    module = ast.parse(code_str, mode='exec')
    last_expr = None
    if module.body and isinstance(module.body[-1], ast.Expr):
        last_expr = module.body[-1]
        module.body = module.body[:-1]
    code_obj = compile(module, filename=f'cell_{execution_count}', mode='exec')

    cell: Dict[str, object] = {}
    cell_outputs = []
    stdout_buf = io.StringIO()
    stderr_buf = io.StringIO()

    plt_module = global_env.get('plt')
    original_show = getattr(plt_module, 'show', None) if plt_module is not None else None

    if plt_module is not None:
        def capture_show(*args, **kwargs):
            fig = plt_module.gcf()
            if fig is None:
                return
            buf = io.BytesIO()
            fig.savefig(buf, format='png', bbox_inches='tight')
            buf.seek(0)
            data = base64.b64encode(buf.read()).decode('ascii')
            cell_outputs.append({
                'output_type': 'display_data',
                'data': {'image/png': data},
                'metadata': {},
            })
            plt_module.close(fig)
        plt_module.show = capture_show

    try:
        with contextlib.redirect_stdout(stdout_buf), contextlib.redirect_stderr(stderr_buf):
            exec(code_obj, global_env)
            if last_expr is not None:
                expr_code = compile(ast.Expression(last_expr.value), filename=f'cell_{execution_count}', mode='eval')
                result = eval(expr_code, global_env)
                if result is not None:
                    data = {'text/plain': repr(result)}
                    html_value = None
                    if hasattr(result, '_repr_html_'):
                        try:
                            html_value = result._repr_html_()
                        except Exception:
                            html_value = None
                    if html_value:
                        data['text/html'] = html_value
                    cell_outputs.append({
                        'output_type': 'execute_result',
                        'execution_count': execution_count,
                        'data': data,
                        'metadata': {},
                    })
    finally:
        if plt_module is not None and original_show is not None:
            plt_module.show = original_show

    stdout_text = stdout_buf.getvalue()
    if stdout_text:
        cell_outputs.insert(0, {
            'output_type': 'stream',
            'name': 'stdout',
            'text': stdout_text,
        })
    stderr_text = stderr_buf.getvalue()
    if stderr_text:
        cell_outputs.append({
            'output_type': 'stream',
            'name': 'stderr',
            'text': stderr_text,
        })

    cell['outputs'] = cell_outputs
    cell['execution_count'] = execution_count
    return cell, global_env, execution_count + 1


demo_code = """
print('Hello from custom executor!')

import numpy as np
values = [v ** 2 for v in range(5)]
values
"""

cell, global_env, next_execution_count = execute_with_capture(demo_code)
cell


{'outputs': [{'output_type': 'stream',
   'name': 'stdout',
   'text': 'Hello from custom executor!\n'},
  {'output_type': 'execute_result',
   'execution_count': 1,
   'data': {'text/plain': '[0, 1, 4, 9, 16]'},
   'metadata': {}}],
 'execution_count': 1}